### Calculating Weights

In [ ]:
import pandas as pd
import geopandas as gpd
from src.config import *
from src.funcs import *
import re

In [ ]:
## Eligible clusters: split across three files covering different regions/additions
elig_rename = {
    'COL5A60A6EACEB1FDF1': 'geom_id', 'COL5A60A6EACE8FA0D5': 'place',
    'COL5A60A6EACEE635C8': 'province', 'COL5A60A6EACEEB6B90': 'population',
    'COL5A60A6EACE23D663': 'proportion', 'COL5A61F9C2A9DB5103': 'strata_description',
    'COL5A61F9C2A950E5C1': 'n_clusters_target', 'COL5A60A6EACE9C5044': 'longitude',
    'COL5A60A6EACE9C9991': 'latitude',
}
eligible_1 = gpd.read_file(DIR_META/"benin sample eligible 1.gpkg").rename(columns=elig_rename)
eligible_2 = gpd.read_file(DIR_META/"benin sample eligible 2.gpkg").rename(columns=elig_rename)
eligible_3 = gpd.read_file(DIR_META/"benin sample eligible 3.gpkg").rename(columns=elig_rename)
settlements_eligible = pd.concat([eligible_1, eligible_2, eligible_3], ignore_index=True) \
                          .drop_duplicates(subset='geom_id', keep='first')

# Houinta (geom_id d7f7f3c8a2cbcd3c5c71397ea658425b7e933ee6, tagged province=Oueme here)
## Actually-surveyed clusters (includes security-driven replacements, with
## population/proportion computed the same way as the frame above).
chosen_rename = {
    '1FFFE3D41E414BF4F43F': 'geom_id', '233E4AB11C414BF4F43F': 'place',
    '32B28DB955414BF4F43F': 'province', '4E0E32C8B9414BF4F43F': 'population',
    '5929598760414BF4F43F': 'proportion', '6350D2B3BE414BF4F43F': 'strata_description',
    '76CC79C927414BF4F43F': 'n_clusters_target', '83A1D65C64414BF4F43F': 'longitude',
    '935C6905CD414BF4F43F': 'latitude',
}
settlements_chosen = gpd.read_file(DIR_META/"Benin Sampling Surveyed.gpkg")
# "Lalo" is duplicated in the gpkg (two placemarks, effectively identical geometry/attributes) --
# drop it so it doesn't double-count respondents via the merge below.
settlements_chosen = settlements_chosen.drop_duplicates(subset='Name', keep='first')
# One cluster is literally named "NA", which read_csv's default NA-value list would silently
# turn into a missing value on any CSV round-trip -- rename it so that never happens.
settlements_chosen['Name'] = settlements_chosen['Name'].replace('NA', 'Unnamed Cluster (NA)')
settlements_chosen = settlements_chosen.rename(columns=chosen_rename).rename(columns={"Name": "sample_number"})

# data/2_cleaned/fem_survey_benin_mapped.csv already has 'location' set to the spatially-joined
# cluster Name (see 02_correct_locations.ipynb), so it lines up with sample_number above.
data=pd.read_csv(DIR_MAPPED_DATA)

#### Pre-process
- clean data types
- remove duplicates
- create household ids
- get number of eligible adults in HH

In [ ]:
## Link population onto settlements_chosen via geom_id, sourced from the original
## sample-draw spreadsheets (the canonical source the gpkg exports were generated from)
## rather than the gpkg's own hashed population column.
sample_draw_v2 = pd.read_csv(DIR_META/"benin_sample_(n=60)-2026-02-11 - sample_draw_v2.csv", header=2)
sample_draw_v21 = pd.read_csv(DIR_META/"benin_sample_(n=60)-2026-02-11 - sample_draw_v2.1.csv", header=2)
population_by_geom_id = pd.concat(
    [sample_draw_v2[['geom_id', 'population']], sample_draw_v21[['geom_id', 'population']]],
    ignore_index=True
).drop_duplicates(subset='geom_id', keep='first').set_index('geom_id')['population']

unmatched = ~settlements_chosen['geom_id'].isin(population_by_geom_id.index)
print(f"Population linked on geom_id for {(~unmatched).sum()}/{len(settlements_chosen)} clusters "
      f"({unmatched.sum()} unmatched)")
if unmatched.any():
    print(settlements_chosen.loc[unmatched, ['sample_number', 'geom_id', 'province']])

settlements_chosen["population"] = settlements_chosen['geom_id'].map(population_by_geom_id)
settlements_chosen["population"]=settlements_chosen["population"].replace({",":""}, regex=True).astype(float)

In [ ]:
# Create HH IDs
data = create_household_id(data)

##### Explore eligible household members

In [ ]:
# Get number of eligible adults in HH listed 
data['eligible_adults_in_hh'] = data.hh_member_index_list.map(parse_n).fillna(1)


In [ ]:
# Check for impossible values
if (data['eligible_adults_in_hh'] < 1).any():
    n_invalid = (data['eligible_adults_in_hh'] < 1).sum()
    print(f"\n⚠️  WARNING: {n_invalid} households with <1 eligible adult")
    print("   This suggests parsing error - please verify column format")
    print("   Setting minimum to 1...")
    data.loc[data['eligible_adults_in_hh'] < 1, 'eligible_adults_in_hh'] = 1

In [ ]:
# Sanity checks
print(f"\nEligible adults per household distribution:")
print(data['eligible_adults_in_hh'].value_counts().sort_index())

print(f"\nMean eligible adults per household: {data['eligible_adults_in_hh'].mean():.2f}")
print(f"Median: {data['eligible_adults_in_hh'].median():.0f}")

#### Stratum weight: with vs without (decide `USE_STRATUM_WEIGHT`)
Stage 1 = PPS cluster weight 1/π_i, with π_i = 1 − (1 − p_i)^k_h. Because it already uses k_h, each province's weighted total comes out proportional to its population. Multiplying by the stratum weight (N_h/N)/(k_h/K) corrects for the allocation a second time: over-sampled provinces end up under-represented. This prints each province's population share, unweighted share, and weighted share with and without the stratum weight; everything else (non-response, Borgou day 1, Plateau language) is identical in both columns.

In [ ]:
stratum_weight_comparison = compare_stratum_weight(settlements_eligible, settlements_chosen, data,
                                                   avg_household_size=5.5, inclusion_probability=PPS_SIZE_MEASURE,
                                                   certainty_units=PPS_CERTAINTY_UNITS)

#### Compute survey weights

In [ ]:
# Settings in src/config.py: PPS_SIZE_MEASURE, USE_STRATUM_WEIGHT, PPS_CERTAINTY_UNITS (see README "Weighting")
data_weighted, cluster_weights = calculate_three_stage_weights(
    settlements_eligible, settlements_chosen, data,
    avg_household_size=5.5, inclusion_probability=PPS_SIZE_MEASURE,
    use_stratum_weight=USE_STRATUM_WEIGHT, certainty_units=PPS_CERTAINTY_UNITS,
    nonresponse_adjustment=NONRESPONSE_ADJUSTMENT)

#### Stage checks
Under PPS, stage 1 × stage 2 ≈ N_h / (k_h × hh_size × m_i), so the cluster population cancels — if stage 1's size measure (`PPS_SIZE_MEASURE`, a column of the eligible frame) is the same quantity as stage 2's population (from the sample-draw sheets). By province, this prints: whether the two match; k_h next to the allocation (`CLUSTERS_ALLOCATED`, reference only); the ratio of w1×w2 to its expected value (~1); certainty clusters (k_h·p_i ≥ 1) and clusters with stage 2 weight < 1; mean stage 3; the Horvitz-Thompson population estimate / frame population; and province shares under w1×w2, w12_expected×w3 and all stages. Per-cluster table (no respondents) → `data/4_validation_checks/02b_stage_weight_check_clusters.csv`.

In [ ]:
stage_check, stage_check_clusters = check_stage_weights(
    cluster_weights, data_weighted, settlements_eligible, avg_household_size=5.5,
    clusters_allocated=CLUSTERS_ALLOCATED, size_col=PPS_SIZE_MEASURE, out_dir=DIR_VALIDATE)

#### Post-stratification (`POSTSTRATIFY_ON`)
Keeps the design-only weight as `normalized_weight_design`. If `POSTSTRATIFY_ON = "province"`, `normalized_weight` becomes w × N_c / N̂_c (N̂_c = sum of design weights in province c, N_c = frame population), renormalised to mean 1 (Statistics Canada, *Survey Methods and Practices*, 7.1.4.1), so downstream notebooks pick it up unchanged. Prints population / sample / design / final shares, the factor per province, thin-cell warnings and Kish DEFF. With `None` it's skipped.

Post-stratifying on province resets each province's total, which cancels the Plateau language adjustment (a uniform multiplier within Plateau). The within-province effects of the draw-attempt rate and Borgou day 1 remain.

In [ ]:
data_weighted, poststrat_summary = apply_poststratification(data_weighted, settlements_eligible, on=POSTSTRATIFY_ON)

Checking which settlements were not actually surveyed

In [ ]:
# sample_number/location are now settlement names (strings), not integer codes
set(settlements_chosen.sample_number.sort_values().unique()) - set(data.location.sort_values().unique())

In [ ]:
data.shape

##### Export

In [ ]:
# NOTE: this used to write to DIR_CLEANED_DATA, which is also 01_mapping.ipynb's raw
# input -- that would have overwritten the source data with the weighted output on
# every run. Writing to its own dedicated file instead.
data_weighted.to_csv(DIR_WEIGHTED_DATA, index=False)

In [ ]:
data_weighted.shape